# Wind → temperature: controlled new-scene transfer

**Teacher / cloud-server edition.** Run in ordinary Jupyter on a CUDA server.
No Colab, private GitHub access, or Google Drive account mounting is used.
This notebook uses actual frozen wind and temperature models, not mock predictions.

This is a NEW experiment on Zhongkai's domain, NOT Yi Qi's original-case reproduction.
The full CUDA experiment has not been executed by the package author. The results
you generate below are required to assess transfer accuracy. See README.md for scope.

**Preparation:** keep this notebook with `code/` and `weights/`. Download
`wind_pollution_reproduction_bundle.zip` from the
[shared data folder](https://drive.google.com/drive/folders/166itkp91W-gNv2L8Qw7hELTU0olf8PCL?usp=drive_link)
and set its local path in Section 2. Only wind assets are extracted.


## 1. Environment and package integrity
Use your cloud provider's CUDA PyTorch image. The optional install cell does not replace PyTorch. Restart the kernel if existing libraries were changed.


In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess
PACKAGE_DIR = Path.cwd().resolve()
if not (PACKAGE_DIR/'code/SOURCE_INFO.json').is_file():
    raise FileNotFoundError('Open the notebook from the extracted wind_temperature_teacher folder.')
def sha_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8 << 20),b''): h.update(b)
    return h.hexdigest()
source_info = json.loads((PACKAGE_DIR/'code/SOURCE_INFO.json').read_text())
for name, expected in source_info['files'].items():
    path = (PACKAGE_DIR/name).resolve()
    if not path.is_relative_to(PACKAGE_DIR) or not path.is_file() or sha_file(path) != expected:
        raise RuntimeError(f'Missing or modified packaged file: {name}')
print('Source and included temperature weight checks passed.')
INSTALL_DEPENDENCIES = False  # Set True once if the following imports fail.
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','-r',str(PACKAGE_DIR/'requirements.txt')])
sys.path.insert(0, str(PACKAGE_DIR/'code'))
import numpy as np, torch, matplotlib.pyplot as plt
import cloud_workflow as wt
import prepare_assets
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| CUDA runtime:',torch.version.cuda, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(), 'GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
else:
    print('CPU can check the temperature model. Full wind forecasting will require CUDA.')


## 2. Local assets and persistent output paths
Change these paths if necessary. The temperature checkpoint is already included.
If the large ZIP has already been extracted, fill `ASSET_OVERRIDES` instead.
Required initial wind: physical `uvw` [3,64,1024,1024], not normalized wind,
a latent, or the small wind-preview figure. The existing W_005000.h5 is suitable.
C_5000 and pollution model weights are NOT needed in this experiment.


In [ ]:
DATA_ARCHIVE = PACKAGE_DIR/'wind_pollution_reproduction_bundle.zip'
ASSET_DIR = PACKAGE_DIR/'wind_assets'      # Use a persistent data disk.
RUN_DIR = PACKAGE_DIR/'runs/temperature_transfer_v1'
# Alternative: fill ALL four keys with your existing trusted local files.
ASSET_OVERRIDES = {
    # 'wind': '/path/to/weights/wind.pt',
    # 'wind_vae': '/path/to/weights/wind_vae.pt',
    # 'geometry': '/path/to/data/inputs/sigma.npy',
    # 'initial_wind': '/path/to/data/physical/W_005000.h5',
}
if ASSET_OVERRIDES:
    if set(ASSET_OVERRIDES) != {'wind','wind_vae','geometry','initial_wind'}:
        raise ValueError('Supply all four ASSET_OVERRIDES, or leave the dictionary empty.')
    paths = {k:Path(v).expanduser().resolve() for k,v in ASSET_OVERRIDES.items()}
else:
    paths = prepare_assets.prepare(DATA_ARCHIVE, ASSET_DIR)
paths['temperature'] = PACKAGE_DIR/'weights/temperature_one_step.pt'
for k,v in paths.items():
    if not v.is_file(): raise FileNotFoundError(f'{k}: {v}')
    print(k, '->', v, f'({v.stat().st_size/2**20:.1f} MiB)')


## 3. The new scenario
Horizontal and vertical spacing are both 4 m after coarsening. The temperature
grid is [16,256,256], NOT Yi Qi's original [64,384,384]. No coordinate claim
about his original city grid is made. Buildings are conservatively coarsened.

Ambient 26 °C, ground/roof 30 °C, and exchange 0.001 /s are chosen controlled
conditions. They are not observations. The 180 s warmup produces three initial
temperature frames; it does not establish equilibrium. Source strengths and
all assumptions are recorded in the run manifest.

**Time:** wind index 5000 is relative t=0. Using the explicit experimental SI
interpretation of dt=0.5, one 50-step wind forecast is 25 s. Temperature forecasts
are 90 s apart. Wind is interpolated to those times, not relabelled as 90 s.
Five temperature steps cover 450 s and require 18 wind-model updates.


In [ ]:
CONFIG = {
    'seed':42, 'factor':4, 'fine_spacing_m':1.0,
    'wind_step_seconds':25.0, 'temperature_step_seconds':90.0,
    'temperature_steps':5,
    'ambient_c':26.0, 'ground_c':30.0, 'roof_c':30.0,
    'surface_exchange_per_s':0.001, 'diffusivity_m2_s':1.0,
    'forcing_layers':2, 'cfl_safety':0.8,
    'temperature_overlap':[4,8,8],
}
manifest = wt.prepare_run(paths, CONFIG, RUN_DIR)
solid = wt.geometry(paths['geometry'])
geo = wt.coarse_geometry(solid, CONFIG['factor'], 4.)
print('Temperature [z,y,x]:', geo['solid'].shape)
print('Solid fraction before/after coarsening:',float(solid.mean()),float(geo['solid'].mean()))
print('Output:', RUN_DIR.resolve())
fig, ax = plt.subplots(1,2,figsize=(11,5),layout='constrained')
ax[0].imshow(solid[10],origin='lower',cmap='gray_r'); ax[0].set_title('Wind geometry at z index 10 (1 m grid)')
ax[1].imshow(geo['solid'][2],origin='lower',cmap='gray_r'); ax[1].set_title('Temperature geometry, height [8,12) m (4 m grid)')
fig.savefig(RUN_DIR/'geometry_preview.png',dpi=150)
plt.show()


## 4. Check the actual temperature weights before the long wind run
This small zero-tensor check tests the neural interface only. It is NOT a physical
temperature result. The actual 15-channel input order is 3 temperature histories,
u/v/w/speed, solid/fluid/roof/height/study/exchange/ground-temperature/ambient.
Roof heat is prescribed in the physical initialization; there is no separate
roof-temperature scalar input in the original one-step U-Net.


In [ ]:
model, stats, patch = wt.load_temperature(paths['temperature'], DEVICE)
print('Training statistics:',stats)
print('Training patch [z,y,x]:',patch)
with torch.inference_mode():
    dummy = torch.zeros((1,15,*patch),device=DEVICE)
    check = model(dummy)
    assert check.shape == (1,1,*patch) and torch.isfinite(check).all()
print('Real weight loading and output-shape check passed. No scientific prediction made here.')
del model, dummy, check
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 5. Forecast and cache the wind
Uses the original wind U-Net and frozen VAE with full-height 256 patches,
32-cell halos and full-domain physical-space stitching. The VAE's sampled
latents and RNG state are saved for restart. No future CFD wind is read.
Only at the temperature interface, convert `(u,v,w)=(-raw_u,raw_v,raw_w)`
to increasing-index coordinates. The sign comes from the NN4PDEs derivative
stencils and is covered by a linear-ramp test; the wind model itself is unchanged.
Each complete update is saved to persistent storage. Do not restart the kernel
while a checkpoint is being written; an interrupted partial file is not reused.


In [ ]:
wt.generate_wind(paths, CONFIG, RUN_DIR, solid, geo, DEVICE)
print('Wind cache ready.')
del solid
gc.collect()


## 6. Generate initial histories and a controlled physical reference
The new solver uses the same predicted wind as the temperature surrogate.
It follows upwind advection + diffusion + surface exchange, with wind-sign-aware
lateral boundaries and a combined explicit stability limit. It is an adaptation
for this new domain, NOT Yi Qi's unchanged physical pipeline and NOT NN4PDEs.

Initial history times are -180,-90,0 s. Future reference frames are +90,...,+450 s.
Wind is held fixed at its frame-start value during each 90 s interval. Each
finished temperature frame is saved. This reference is a numerical comparison,
not real measurements or a validation of the upstream wind model.


In [ ]:
times = wt.physical_reference(CONFIG, RUN_DIR, geo, DEVICE)
print('Saved reference times (s):', times)
print('Initial history files:', [str(RUN_DIR/'temperature'/f'reference_{i:03d}.npy') for i in range(3)])


## 7. Coupled prediction and diagnostic comparisons
**recursive:** uses only the common initial history, then its own temperature predictions.
**one_step:** gives the model physical histories at each time, isolating one-step transfer error.
**persistence:** holds the temperature at t=0 unchanged.

Future physical temperatures are never input to the recursive branch. Model
statistics are loaded from its checkpoint; output is restored to °C. Inference
uses [8,32,32] patches with averaged overlap, not the wind core/halo scheme.
GroupNorm means tiled and full-volume inference are not mathematically identical.
Predictions are not clipped or boundary-projected to hide transfer errors.


In [ ]:
rows = wt.run_temperature(paths, CONFIG, RUN_DIR, geo, DEVICE)
import csv
with (RUN_DIR/'temperature/metrics.csv').open('w',newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
for row in rows:
    print(f"+{row['time_seconds']:3d} s | {row['mode']:11s} | MAE {row['mae_c']:.4f} °C | RMSE {row['rmse_c']:.4f} °C | bias {row['bias_c']:+.4f} °C")


## 8. Inspect the results
All temperature panels share one colour range. Difference panels share a symmetric
range around zero; red means prediction is hotter, blue means colder. Grey marks
solid cells, excluded from metrics. Height 10 m selects the [8,12) m coarse cell.

If one_step is poor, the model does not transfer well even with physical histories.
If recursive is much worse than one_step, error accumulation is important.
Compare against persistence before claiming that the coupled forecast is useful.


In [ ]:
wt.plot_results(CONFIG, RUN_DIR, geo, height_m=10.)
print('Report these files, together with manifest.json:')
for name in ('figures/temperature_comparison.png','figures/temperature_mae.png','temperature/metrics.csv'):
    print(RUN_DIR/name)


## Restart / share
Rerun with the same paths, configuration and environment to continue from saved
wind and temperature stages. Use a new `RUN_DIR` for a changed experiment.
Keep persistent storage when releasing the cloud instance.

To share results, send `manifest.json`, the metrics CSV, and the two figures.
Full 3D frames are saved separately. Running this notebook does not upload results,
modify the shared Git repository, retrain any model, or use private credentials.
